# Automating Employee Quarterly Check-in Process with Workflow

The Quarterly Check-in AI Agent aims to streamline the employee-manager check-in process, offering guided workflows for employees and managers to track performance, goals, and development needs. This AI agent can reduce the administrative burden of check-ins, ensure meaningful feedback, and align individual performance with organizational objectives.

In [1]:
from IPython.display import Markdown
from slackagents import Assistant, WorkflowAgent, FunctionTool
from slackagents.graph.execution_graph import ExecutionGraph, ExecutionTransition

## Data Agent

The Data Agent is responsible for collecting and storing the data required for the AI Agent to function. The Data Agent is responsible for the following tasks:
- Loading an employee's Jira record
- Asking questions to the employee to collect information for the report
- Generating a report for the employee
- Writing the report to the employee's local directory

### Define tools for the Data Agent
The Data Agent has the following tools:
- `load_jira_record_tool`: Load an employee's Jira record
- `write_tool`: Write the report to the employee's local directory

In [2]:
# Define the tools for the agent
def load_jira_record_tool(employee_id: str):
    """Load an employee's Jira performance management record
    
    :param employee_id: The employee ID
    :type employee_id: string
    :return: The employee's Jira performance management record
    :rtype: string
    """
    try:
        with open(f"assets/employee_jira_{employee_id}.md", "r") as f:
            return f.read()
    except FileNotFoundError:
        return "Jira record not found"
    
def write_tool(content: str, document_name: str):
    """Write content as a markdown file to the employee's local directory
    
    :param content: The content to be written
    :type content: string
    :param document_name: The name of the document to be written to
    :type document_name: string 
    """
    with open(f"assets/{document_name}", "w") as f:
        f.write(content)
    return f"Successfully saved the report to {document_name}"

### Create the Data Agent
We will create the Data Agent with the following system prompt:

In [3]:
system_prompt = """\
# Quarterly Check-in Workflow - Data Agent
You are an AI agent designed to help employees summarize the employee's progress against their set goals by generating a report. 

## Load Jira Record
You can first load an employee's Jira performance management record to help generate the report. 

## Information Collection
You should review the employee's progress in Jira against their set goals. 

Then, you should ask questions to the employee, such as the following, to help collect information for the report:

"What were your key accomplishments this quarter?"
"What challenges did you face?"
"What areas do you think need improvement?"
"What are your goals for the next quarter?"
"Do you need additional resources or support?"

In the end, you should create notes with checklists of the progress vs goals for the employee to review their progress and prepare for the check-in meeting. 

## Report Generation

Now, generate a report for the employee to prepare for the check-in meeting. In the report, you should include the following sections in order:
1. Title. Below the title, you should include the employee and their manager information.
2. A checklist table of progress against goals should be at the first section.
3. Sections for breakdowns of Goals and Progress, Challenges, Improvement, etc.

You should return the generated report to ask the employee for feedback and revisions, before saving the report to the employee's local directory. 

## Report Saving
After the employee approves the report, you should save the report to the employee's local directory as a markdown file with the following naming convention: `Quarterly_Check-in_Report_<quarter>_<year>_<employee_name>.md`.

## Automatic Transition
Whenever the transition condition is met, you should transition to the next module automatically.
"""

data_agent = Assistant(
    name="Data Agent",
    desc="AI agent designed to load an employee's Jira record and generate a report for the quarterly check-in meeting. ",
    tools = [
        FunctionTool.from_function(load_jira_record_tool),
        FunctionTool.from_function(write_tool),
    ],
    system_prompt=system_prompt,
    verbose=True
)

Next, let's define the Calendar Agent:

## Calendar Agent
The Calendar Agent is responsible for finding the best time slots for the meeting and sending the calendar invites. The Calendar Agent has the following tools:
- `load_employee_calendar_tool`: Load an employee's calendar for the quarter
- `send_calendar_invite_tool`: Send the calendar invites to the employees

### Define tools for the Calendar Agent

In [4]:
def load_employee_calendar_tool(employee_id: str):
    """Load an employee's calendar for the quarter
    
    :param employee_id: The employee ID
    :type employee_id: string
    :return: The employee's calendar for the quarter
    :rtype: string
    """
    with open(f"assets/employee_calendar_{employee_id}.md", "r") as f:
        return f.read()

def send_calendar_invite_tool(employee_id: str, time_slot: str, meeting_title: str, notes: str):
    """Send a calendar invite to multiple employees
    
    :param employee_id: The employee ID
    :type employee_id: string
    :param time_slot: The time slot to send the invite to
    :type time_slot: string
    :param meeting_title: The title of the meeting
    :type meeting_title: string
    :param notes: The notes of the meeting
    :type notes: string
    """
    return f"Successfully scheduled the meeting with {employee_id} at {time_slot} with title {meeting_title} and notes {notes}"


### Create the Calendar Agent

Calendar Agent's system prompt:

In [5]:
system_prompt = """\
# Quarterly Check-in Workflow - Calendar Agent
You are an AI agent designed to load an multiple employees' calendars, find the best time slots for the meeting, and send the calendar invites.

### Information Collection
You should first load the employees' calendars by calling the `load_employee_calendar_tool` tool. 

### Time Slot Selection
Then, you should find the best time slots for the meeting from the calendars. If there are multiple approvable time slots, you should return all of them and let the employee choose one. 
You should rank the time slots by the following criteria:
1. The time slot should be during the business hours.
2. The time slot should have the least number of meetings.
3. The time slot should be during the employee's working hours.
4. Morning slots are generally better than afternoon slots.

### Calendar Invite Sending
After you find the best time slots, you should send the calendar invites to the employees by calling the `send_calendar_invite_tool` tool.

### Automatic Transition
Whenever the transition condition is met, you should transition to the next module automatically.
"""

calendar_agent = Assistant(
    name="Calendar Agent",
    desc="AI agent designed to load an employee's calendar and send the calendar invites",
    tools=[
        FunctionTool.from_function(load_employee_calendar_tool),
        FunctionTool.from_function(send_calendar_invite_tool)
    ],
    system_prompt=system_prompt,
    verbose=True
)

Finally, let's define the Email Agent:

## Email Agent

Email Agent is the last step in the workflow. It is responsible for sending the emails to the employees to inform them about the meeting.

### Define tool for the Email Agent

- `send_email_tool`: Send an email to an employee

In [6]:
def send_email_tool(employee_id: str, subject: str, body: str):
    """Send an email to an employee

    :param employee_id: The employee ID
    :type employee_id: string
    :param subject: The subject of the email
    :type subject: string
    :param body: The body of the email
    :type body: string
    """
    return f"Successfully sent an email to {employee_id} with subject {subject} and body {body}"

### Create the Email Agent

Email Agent's system prompt:

In [7]:
system_prompt = """\
# Quarterly Check-in Workflow - Email Agent
You are an AI agent designed to send emails to employees.
"""

email_agent = Assistant(
    name="Email Agent",
    desc="AI agent designed to send emails to employees",
    tools=[FunctionTool.from_function(send_email_tool)],
    system_prompt=system_prompt,
    verbose=True
)

## Define the transitions for the workflow
Now, let's define the transitions for the workflow. The workflow is as follows:
1. The Data Agent loads the employee's Jira record and generates a report.
2. The Calendar Agent finds the best time slots for the meeting and sends the calendar invites.
3. The Email Agent sends the emails to the employees to inform them about the meeting.


In [8]:
## Composer
graph = ExecutionGraph()
graph.add_agent(data_agent)
graph.add_agent(calendar_agent)
graph.add_agent(email_agent)

graph.add_transition(
    ExecutionTransition(
        source_module=graph.get_module("Data Agent"), 
        target_module=graph.get_module("Calendar Agent"), 
        desc="When the report written to the employee's local directory, you should schedule the meeting with the employees."
    )
)

graph.add_transition(
    ExecutionTransition(
        source_module=graph.get_module("Calendar Agent"), 
        target_module=graph.get_module("Email Agent"), 
        desc="When the meeting is scheduled, you should send the calendar invites to the employees."
    )
)

graph.set_initial_module(graph.get_module("Data Agent"))

quarterly_checkin_agent = WorkflowAgent(
    name="Quarterly Check-in Workflow",
    desc="Workflow designed to automate the employee quarterly check-in process",
    graph=graph,
    verbose=True
)

## Example interaction
 
Users can start the workflow by sending a message to the composer. For example:

In [ ]:
message = "The quarterly check-in is due in a week. Please help prepare for the check-in."
response = quarterly_checkin_agent.chat(message)
display(Markdown(f"User: {message}\n\nResponse: {response}"))